# Test model on other datasets

This code is just provided to easily check the performance of a model trained on a smaller dataset with respect to a larger one. You can test the transferability here.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ase.io
from tqdm import tqdm
from icet import ClusterExpansion

chemical_symbols = ("Cu", "Au")
base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)

max_atom_num = 8
test_set_num = 10
test_set_path = (
    struct_path / f"enumerated_structures_{test_set_num}_calculated.extxyz"
)
test_set = ase.io.read(test_set_path, index=":")
ce = ClusterExpansion.read(struct_path / f"ce_model_{max_atom_num}.ce")
ref_energies = []
model_energies = []
for atoms in tqdm(test_set):
    model_pred = ce.predict(atoms)
    ref_energies.append(atoms.info["mixing_energy"])
    model_energies.append(model_pred)

ref_energies = np.asarray(ref_energies)
model_energies = np.asarray(model_energies)
RMSE = np.sqrt(np.mean((ref_energies - model_energies) ** 2))
print(f"RMSE: {RMSE}")
MAE = np.mean(np.abs(ref_energies - model_energies))
print(f"MAE: {MAE}")

plt.scatter(ref_energies, model_energies)
plt.axline([np.min(ref_energies), np.min(ref_energies)], slope=1, color="k")
plt.xlabel("Reference energy / eV/atom")
plt.ylabel("Predicted energy / eV/atom")
plt.show()